<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Calibration images, 8×6 internal-corner geometry, 0.03 m square size |
| **Final outputs** | Intrinsic matrix K, one (R,t) pose per retained view, reprojection metrics, six diagnostic figures |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Open the lab with the project environment selected in VS Code.
2. Treat this notebook as the coding plan for `main.ipynb`; do not compute results here.
3. Proceed only in the order below because later calibration quantities depend on earlier validated geometry.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create the Setup

**Purpose**

Fix the calibration geometry and numerical environment.

**Inputs**

Python environment; 8 by 6 internal corners.

**Procedure**

1. Import `Path`, OpenCV, NumPy, and Matplotlib.
2. Set NumPy display precision so matrices and residuals are readable.
3. Define square size, 8×6 internal corners, minimum valid views, and corner-refinement criteria.

**Outputs**

Imports; 0.03 m spacing; minimum 3 views.

**Validation**

Check the board uses internal corners, not numbers of squares.

**On failure**

Stop on missing dependencies or an incorrect board specification.

## 2. Create Repository-Relative Paths

**Purpose**

Keep the experiment portable within the lab.

**Inputs**

Lab notebooks directory.

**Procedure**

1. Create the calibration-image path and `outputs/figures/` path.
2. Create the output directory if needed.
3. Define supported image extensions; use no machine-specific absolute path.

**Outputs**

DATA_DIR and OUTPUT_DIR paths.

**Validation**

Input directory exists; output directory is writable.

**On failure**

Resolve the working directory before loading data.

## 3. Discover and Validate Calibration Images

**Purpose**

Build a deterministic image inventory.

**Inputs**

Supported files under data/calibration_images.

**Procedure**

1. Enumerate supported image files and sort them deterministically.
2. If no image is found, stop with `FileNotFoundError`.
3. Create containers for readable images and retained calibration views.

**Outputs**

Sorted paths and readable_calibration_views.

**Validation**

At least one readable image; retain names alongside arrays.

**On failure**

Log and skip unreadable files; stop if none remain.

## 4. Detect and Refine Chessboard Corners

**Purpose**

Obtain consistent subpixel correspondences.

**Inputs**

Readable BGR images; REFINE_CRITERIA.

**Procedure**

1. For each image: read it and convert it to grayscale.
2. Detect the complete 8×6 chessboard pattern.
3. If found, refine corners with `cornerSubPix`; otherwise reject that view.
4. Store image name, image array, and refined 2-D points.
5. Stop if fewer than the required number of valid views remain.

**Outputs**

views with im_pts shaped (48, 2), in pixels.

**Validation**

Complete 8 by 6 pattern; at least 3 retained views.

**On failure**

Skip incomplete detections explicitly; stop below MIN_VALID_VIEWS.

## 5. Build Metric Planar Coordinates

**Purpose**

Attach metric board coordinates to detected corners.

**Inputs**

Board dimensions and SQUARE_SIZE_M.

**Procedure**

1. Create 48 planar points in the same ordering as detected corners.
2. Set each point to `(x·square_size, y·square_size)` in metres.
3. Keep the board on `Z=0`; add the zero z-coordinate only when 3-D points are needed.

**Outputs**

plane_pts shaped (48, 2), in metres.

**Validation**

x varies fastest, matching corner ordering; board lies on Z=0.

**On failure**

Resolve ordering or units before estimating homographies.

## 6. Define Homogeneous-Coordinate Conversion

**Purpose**

Enable homogeneous matrix multiplication.

**Inputs**

Plane/image points shaped (N, 2).

**Procedure**

1. Create a helper that appends a homogeneous coordinate of 1 to every 2-D point.
2. Use it for both plane and image correspondences.
3. Validate the returned shape before matrix computations.

**Outputs**

Homogeneous arrays shaped (N, 3).

**Validation**

Last column equals 1; first two columns preserve coordinates.

**On failure**

Do not pass arrays with missing or extra coordinate columns.

## 7. Normalize Plane and Image Points

**Purpose**

Balance coordinate magnitudes before solving DLT.

**Inputs**

48 plane and image points per view.

**Procedure**

1. Compute each point-set centroid and shift to zero mean.
2. Compute mean Euclidean distance from the centroid.
3. Set scale to `sqrt(2)/mean_distance`; reject coincident-point configurations.
4. Build each 3×3 normalization transform and apply it.
5. Store `T_plane`, `T_image`, and normalized point sets per view.

**Outputs**

T_plane, T_image and normalized points.

**Validation**

Centroid approximately zero; mean radius approximately sqrt(2). Current coincidence guard: mean distance below 1e-12.

**On failure**

Stop on coincident points; a large normalization scale alone is not degeneracy.

## 8. Build the Normalized DLT System

**Purpose**

Estimate a planar mapping up to scale.

**Inputs**

Normalized correspondences.

**Procedure**

1. For every correspondence add the two standard DLT rows to matrix `Q`.
2. Solve `Qh=0` by SVD.
3. Select the right-singular vector for the smallest singular value.
4. Reshape it into a normalized 3×3 homography.

**Outputs**

Q shaped (96, 9); singular values; H_normalized shaped (3, 3).

**Validation**

Finite Q and homography; use the last right-singular vector. An exact solution is expected to have a near-zero last singular value.

**On failure**

Inspect degenerate geometry if the solution is not isolated; do not reject an exact fit because sigma_max/sigma_min is large.

## 9. Denormalize Every Homography

**Purpose**

Express the homography in metres-to-pixels coordinates.

**Inputs**

H_normalized and both normalization matrices.

**Procedure**

1. Compute `H = inv(T_image) @ H_normalized @ T_plane`.
2. Normalize by `H[2,2]`; reject numerically degenerate scale.
3. Store one finite 3×3 homography per retained view.
4. Recheck that each view still contains 48 plane/image correspondences.

**Outputs**

One H shaped (3, 3) per view.

**Validation**

Current helper rejects abs(H[2,2]) below 1e-12 before scaling; verify 48 paired points.

**On failure**

Stop on the helper's scale error. This is a chosen normalization limitation, not proof the plane is parallel to the image.

## 10. Build Zhang Multi-View Constraints

**Purpose**

Combine orthogonality constraints from multiple views.

**Inputs**

Retained homographies.

**Procedure**

1. Extract `h1` and `h2` from every homography.
2. Implement the Zhang `v_ij` vector.
3. Stack `v12` and `v11-v22` for all views into matrix `V`.
4. Solve `Vb=0` by SVD and retain the last right-singular vector `b`.

**Outputs**

V shaped (2*number_of_views, 6); unit-norm b shaped (6,).

**Validation**

Inspect the singular spectrum and residual in the same scaling as V. Current implementation stacks unnormalized rows.

**On failure**

Do not infer failure from an absolute residual alone or require noisy data to reach machine precision.

## 11. Recover the Intrinsic Matrix

**Purpose**

Recover the camera's intrinsic parameters.

**Inputs**

b in order (B11,B12,B22,B13,B23,B33).

**Procedure**

1. Unpack `b` into the six symmetric-conic coefficients in the order used by `build_zhang_constraint_vector`.
2. Compute `denominator = b11*b22-b12**2`; the current code raises when its absolute value is below `1e-12`.
3. Compute `v0` and `lambda_`, then the two focal-length radicands.
4. Describe the existing sign-retry branch accurately: it negates `b` and recomputes the expressions, but the radicands are mathematically invariant to a global sign flip.
5. Assemble `K` with the closed-form formulas. Review finiteness and positive focal lengths before interpreting the result; the limitations section distinguishes this review from the current automatic checks.

**Outputs**

alpha, beta, gamma, u0, v0 and K shaped (3, 3).

**Validation**

Review finite radicands and positive focal lengths. Existing final check verifies only K shape and finiteness.

**On failure**

Stop analysis if K is nonphysical; sign reversal cannot repair negative radicands, whose ratios are sign-invariant. See implementation limits below.

## 12. Recover One Camera Pose per View

**Purpose**

Recover board-to-camera orientation and translation.

**Inputs**

K and each H.

**Procedure**

1. Compute `K^-1 h1`, `K^-1 h2`, and `K^-1 h3`.
2. Use the norm of `K^-1 h1` for the common pose scale.
3. Form `r1`, `r2`, `r3=r1×r2`, and `t`.
4. Project the approximate rotation to the nearest orthonormal matrix with SVD.
5. If `det(R)<0`, correct the sign so `R` is a proper rotation.
6. Store finite `R` and `t`.

**Outputs**

R shaped (3, 3); t shaped (3,), in metres.

**Validation**

Current scale is 1/norm(K^-1 h1); SVD projection enforces a proper rotation. Check orthogonality and determinant near +1.

**On failure**

Do not describe t_z>0 as enforced: current code has no cheirality correction.

## 13. Reproject Calibration Points

**Purpose**

Predict image coordinates from the recovered camera.

**Inputs**

K, R, t; board coordinates on Z=0.

**Procedure**

1. Convert planar points to `(X,Y,0)`.
2. Transform them into the camera frame with `R` and `t`.
3. Project with `K` and divide by homogeneous depth.
4. Reject near-zero depth and store the projected 2-D points.

**Outputs**

world_xyz shaped (48, 3); projected shaped (48, 2).

**Validation**

Current guard checks absolute homogeneous depth against 1e-12. Review positive camera-frame depth separately.

**On failure**

Stop on near-zero depth; flag negative depths as physically invalid even if division is finite.

## 14. Compute Calibration Errors

**Purpose**

Measure calibration fit in pixel units.

**Inputs**

Observed and projected points with identical ordering.

**Procedure**

1. Compute residual vectors `projected-observed`.
2. Convert residuals to point-wise Euclidean errors.
3. Compute per-view mean error and RMSE.
4. Concatenate all errors and compute overall mean error and RMSE.

**Outputs**

Per-point errors; per-view and global mean/RMSE.

**Validation**

Euclidean error per point; RMSE=sqrt(mean(error**2)); global metrics pool all points.

**On failure**

Investigate nonfinite values or systematic residuals; no universal pixel acceptance threshold is implemented.

## 15. Generate the Required Diagnostics

**Purpose**

Expose detection, geometry and residual quality visually.

**Inputs**

Stored views, K, poses and error arrays.

**Procedure**

1. Save the homography-estimation pipeline figure.
2. Save detected corners, camera poses, and reprojection overlays.
3. Save mean reprojection error by view and the global error distribution.
4. Build every plot from the same validated arrays used for numerical metrics.

**Outputs**

Six PNG diagnostics listed in the output contract.

**Validation**

Overlays and plots must use the same retained views and errors; camera centre is -R.T@t.

**On failure**

Regenerate missing plots; do not treat visually plausible overlays as physical validation.

## 16. Run Final Validation

**Purpose**

Check the implemented numerical and artifact conditions.

**Inputs**

Computed calibration and REQUIRED_OUTPUTS.

**Procedure**

1. Follow the final validation cell in `main.ipynb`: check finite `K` with shape `(3,3)`.
2. Check finite homographies, rotations with `R.T @ R` close to identity and determinant close to `+1`, and finite error arrays.
3. Check that all six declared figures exist.
4. Interpret the success message as the result of these implemented checks. Positive depths, finite translations and reprojection-quality thresholds are additional review requirements, not checks currently enforced by that cell.

**Outputs**

Final validation outcome.

**Validation**

K finite 3 by 3; H finite; R.T@R near identity and det(R) near 1; finite errors; six files present.

**On failure**

Any existing failed check stops execution; separately report unimplemented physical checks.

## Complete End-to-End Pseudocode

This is control-flow pseudocode, not an additional Python API. It follows the current implementation; review requirements and known gaps are distinguished below.

```text
configure board=(8,6), spacing=0.03 m, minimum_views=3
resolve data and figure paths
read sorted supported files; log unreadable images
for each readable image:
    points <- detect_and_refine_chessboard_corners(image)
    if no complete pattern: log rejection and continue
    retain image, name, points
if retained views < minimum_views: stop
plane <- build_chessboard_plane_points()
for each retained view:
    append homogeneous coordinates to plane and image points
    compute T_plane and T_image; reject coincident point sets
    normalize points; Q <- build_homography_dlt_matrix(...)
    H_normalized <- reshape(last right-singular vector of Q, 3, 3)
    H <- denormalize_homography_matrix(H_normalized, T_image, T_plane)
V <- stack build_zhang_constraints_for_view(H) for all views
b <- last right-singular vector of V; report spectrum and residual
compute intrinsic denominator; stop if current scale guard fails
compute v0 and lambda; current code retries with -b if radicands are nonpositive
form K using the closed-form expressions
for each retained view:
    R, t <- recover_camera_pose(K, H)
    camera_points <- world_points @ R.T + t
    homogeneous_pixels <- camera_points @ K.T
    if any absolute depth < 1e-12: stop
    projected <- divide by homogeneous depth
    errors <- Euclidean distances to observed points
pool point errors; calculate global mean and RMSE
save the six diagnostics using those arrays
run current K/H/rotation/error/file checks; stop if any fail
report numerical completion and the physical-validation limits below
```

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Load the Sorted JPEG Calibration Images
- **Task 2:** Detect and Refine Chessboard Corners
- **Task 3:** Build the Planar World Coordinates
- **Task 4:** Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$
- **Task 5:** Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD
- **Task 6:** Denormalize Each Homography
- **Task 7:** Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD
- **Task 8:** Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$
- **Task 9:** Recover $R$ and $t$ for Every Retained View
- **Task 10:** Reproject the $Z=0$ Calibration Points
- **Task 11:** Compute Point-wise Errors, Mean Error and RMSE
- **Task 12:** Produce and Save the Six Required Diagnostic Figures
- **Task 13:** Run the Numerical and Output-file Validation Checks


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation** of this algorithm.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 6 files under `../outputs/figures/`:

- `detected_chessboard_corners.png`
- `homography_estimation_pipeline.png`
- `estimated_camera_poses.png`
- `reprojection_results.png`
- `mean_reprojection_error_by_view.png`
- `reprojection_error_distribution.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/computer-vision/camera-calibration/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `detect_and_refine_chessboard_corners`
- `build_chessboard_plane_points`
- `convert_to_homogeneous_coordinates`
- `compute_point_normalization_transform`
- `build_homography_dlt_matrix`
- `denormalize_homography_matrix`
- `build_zhang_constraint_vector`
- `build_zhang_constraints_for_view`
- `recover_camera_pose`

## Implementation Limits and Interpretation

- The implementation uses normalized DLT points but **does not row-normalize Zhang's V** or apply spectral-gap rejection. Row normalization changes equation weighting; it is not proof of physical correctness and does not force noisy residuals to machine precision.
- A near-zero final singular value is expected for exact homogeneous systems. If conditioning diagnostics are added later, distinguish the nullspace gap from conditioning in the constrained subspace; do not reject an exact fit using `sigma_max/sigma_min > 1e6`.
- The intrinsic denominator threshold is absolute and scale-dependent. Global sign reversal leaves the closed-form radicands unchanged. No Cholesky extraction or automatic repair of an indefinite conic is present.
- Rotation projection is implemented, including determinant correction. The in-place update of the local SVD factor is valid; it is not a demonstrated NumPy bug.
- Positive focal lengths, positive depth for **every** board point and finite translations require review. The current code does not enforce cheirality, and `t_z>0` alone would not prove all board points are in front.
- The model excludes lens distortion and nonlinear refinement. No universal RMSE acceptance threshold, JSON audit export or new package/module is introduced by this algorithm update.

## 17. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.